In [1]:
!pip install sentence-transformers==4.1.0 | tail -n 1

In [1]:
import math

import numpy as np
import scipy
import torch
from sentence_transformers import SentenceTransformer

In [2]:
# Example documents
documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.'
]

In [4]:
# Load a pre-trained model
model = SentenceTransformer('sentence-transformers/paraphrase-MiniLM-L6-v2')

In [5]:
# Generate embeddings
embeddings = model.encode(documents)

In [6]:
embeddings.shape

(4, 384)

In [7]:
embeddings

array([[-0.22804348, -0.24647684, -0.00319294, ...,  0.45528147,
         0.6341976 ,  0.5375049 ],
       [-0.35791597, -0.32083988,  0.15963267, ..., -0.07050666,
         0.92750263,  0.34377286],
       [ 0.20302926, -0.26898623,  0.1628511 , ..., -0.19650966,
        -0.03379833,  0.5956154 ],
       [-0.04264304, -0.45721593, -0.09526499, ..., -0.5803074 ,
         0.17248419,  0.09127873]], shape=(4, 384), dtype=float32)

In [8]:
def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)

In [9]:
euclidean_distance_fn(embeddings[0], embeddings[1])

5.961789851413909

In [11]:
euclidean_distance_fn(embeddings[1], embeddings[0])

5.961789851413909

In [12]:
l2_dist_manual = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])

l2_dist_manual

array([[0.        , 5.96178985, 7.33939781, 7.15578276],
       [5.96178985, 0.        , 7.76861601, 7.39359048],
       [7.33939781, 7.76861601, 0.        , 5.91992735],
       [7.15578276, 7.39359048, 5.91992735, 0.        ]])

In [13]:
l2_dist_manual[0,1]

np.float64(5.961789851413909)

In [14]:
l2_dist_manual[1,0]

np.float64(5.961789851413909)

In [15]:
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
l2_dist_scipy

array([[0.        , 5.96178961, 7.33939853, 7.15578165],
       [5.96178961, 0.        , 7.76861598, 7.39359146],
       [7.33939853, 7.76861598, 0.        , 5.9199269 ],
       [7.15578165, 7.39359146, 5.9199269 , 0.        ]])

In [16]:
np.allclose(l2_dist_manual, l2_dist_scipy)

True

In [17]:
def dot_product_fn(vector1, vector2):
    return sum(x * y for x, y in zip(vector1, vector2))

In [18]:
dot_product_fn(embeddings[0], embeddings[1])

np.float32(18.535397)

In [19]:
dot_product_manual = np.empty([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        dot_product_manual[i,j] = dot_product_fn(embeddings[i], embeddings[j])

dot_product_manual

array([[33.74439621, 18.53539658,  8.56981564,  7.83093357],
       [18.53539658, 38.86933136,  7.88997412,  8.66340351],
       [ 8.56981564,  7.88997412, 37.26199722, 17.66957474],
       [ 7.83093357,  8.66340351, 17.66957474, 33.12267303]])

In [20]:
embeddings

array([[-0.22804348, -0.24647684, -0.00319294, ...,  0.45528147,
         0.6341976 ,  0.5375049 ],
       [-0.35791597, -0.32083988,  0.15963267, ..., -0.07050666,
         0.92750263,  0.34377286],
       [ 0.20302926, -0.26898623,  0.1628511 , ..., -0.19650966,
        -0.03379833,  0.5956154 ],
       [-0.04264304, -0.45721593, -0.09526499, ..., -0.5803074 ,
         0.17248419,  0.09127873]], shape=(4, 384), dtype=float32)

In [21]:
# Matrix multiplication operator
dot_product_operator = embeddings @ embeddings.T
dot_product_operator

array([[33.744396 , 18.535397 ,  8.569815 ,  7.8309336],
       [18.535397 , 38.869335 ,  7.889973 ,  8.6634035],
       [ 8.569815 ,  7.889973 , 37.261997 , 17.669575 ],
       [ 7.8309336,  8.6634035, 17.669575 , 33.122677 ]], dtype=float32)

In [22]:
np.allclose(dot_product_manual, dot_product_operator, atol=1e-05)

True

In [23]:
# L2 norms
l2_norms = np.sqrt(np.sum(embeddings**2, axis=1))
l2_norms

array([5.808994 , 6.2345276, 6.1042604, 5.75523  ], dtype=float32)

In [24]:
# L2 norms reshaped
l2_norms_reshaped = l2_norms.reshape(-1,1)
l2_norms_reshaped

array([[5.808994 ],
       [6.2345276],
       [6.1042604],
       [5.75523  ]], dtype=float32)

In [25]:
normalized_embeddings_manual = embeddings/l2_norms_reshaped
normalized_embeddings_manual

array([[-0.03925697, -0.04243021, -0.00054966, ...,  0.07837527,
         0.10917512,  0.09252978],
       [-0.05740868, -0.05146178,  0.02560461, ..., -0.01130906,
         0.14876871,  0.05514016],
       [ 0.03326026, -0.04406533,  0.02667827, ..., -0.03219222,
        -0.00553684,  0.09757372],
       [-0.00740944, -0.07944356, -0.01655277, ..., -0.10083131,
         0.02996999,  0.01586014]], shape=(4, 384), dtype=float32)

In [26]:
np.sqrt(np.sum(normalized_embeddings_manual**2, axis=1))

array([0.99999994, 0.99999994, 1.        , 1.        ], dtype=float32)

In [27]:
normalized_embeddings_torch = torch.nn.functional.normalize(
    torch.from_numpy(embeddings)
).numpy()
normalized_embeddings_torch

array([[-0.03925697, -0.04243021, -0.00054966, ...,  0.07837527,
         0.10917512,  0.09252978],
       [-0.05740868, -0.05146178,  0.02560461, ..., -0.01130906,
         0.14876871,  0.05514016],
       [ 0.03326026, -0.04406533,  0.02667827, ..., -0.03219222,
        -0.00553684,  0.09757372],
       [-0.00740944, -0.07944355, -0.01655277, ..., -0.10083131,
         0.02996999,  0.01586013]], shape=(4, 384), dtype=float32)

In [28]:
np.allclose(normalized_embeddings_manual, normalized_embeddings_torch)

True

In [29]:
dot_product_fn(normalized_embeddings_manual[0], normalized_embeddings_manual[1])

np.float32(0.51179683)

In [30]:
cosine_similarity_manual = np.empty([4,4])
for i in range(normalized_embeddings_manual.shape[0]):
    for j in range(normalized_embeddings_manual.shape[0]):
        cosine_similarity_manual[i,j] = dot_product_fn(
            normalized_embeddings_manual[i], 
            normalized_embeddings_manual[j]
        )

cosine_similarity_manual

array([[1.00000012, 0.51179683, 0.24167828, 0.23423405],
       [0.51179683, 1.00000024, 0.20731883, 0.24144743],
       [0.24167828, 0.20731883, 1.00000095, 0.50295621],
       [0.23423405, 0.24144743, 0.50295621, 1.00000024]])

In [31]:
cosine_similarity_operator = normalized_embeddings_manual @ normalized_embeddings_manual.T
cosine_similarity_operator

array([[1.0000001 , 0.51179683, 0.24167825, 0.23423405],
       [0.51179683, 1.0000002 , 0.20731883, 0.24144743],
       [0.24167825, 0.20731883, 1.000001  , 0.50295615],
       [0.23423405, 0.24144743, 0.50295615, 1.0000002 ]], dtype=float32)

In [32]:
np.allclose(cosine_similarity_manual, cosine_similarity_operator)

True

In [33]:
1 - cosine_similarity_manual

array([[-1.19209290e-07,  4.88203168e-01,  7.58321717e-01,
         7.65765950e-01],
       [ 4.88203168e-01, -2.38418579e-07,  7.92681172e-01,
         7.58552566e-01],
       [ 7.58321717e-01,  7.92681172e-01, -9.53674316e-07,
         4.97043788e-01],
       [ 7.65765950e-01,  7.58552566e-01,  4.97043788e-01,
        -2.38418579e-07]])

In [34]:
# First, embed the query:
query_embedding = model.encode(
    ["Who is responsible for a coding project and fixing others' mistakes?"]
)

# Second, normalize the query embedding:
normalized_query_embedding = torch.nn.functional.normalize(
    torch.from_numpy(query_embedding)
).numpy()

# Third, calculate the cosine similarity between the documents and the query by using the dot product:
cosine_similarity_q3 = normalized_embeddings_manual @ normalized_query_embedding.T

# Fourth, find the position of the vector with the highest cosine similarity:
highest_cossim_position = cosine_similarity_q3.argmax()

# Fifth, find the document in that position in the `documents` array:
documents[highest_cossim_position]

# As you can see, the query retrieved the document `Bugs introduced by the intern had to be squashed by the lead developer.` which is what we would expect.

'Bugs introduced by the intern had to be squashed by the lead developer.'